In [50]:
import pandas as pd

In [51]:
data = pd.read_csv("house_data.csv")

In [52]:
data.shape

(1000, 9)

In [53]:
data.head(5)

,sqft,bedrooms,bathrooms,age,floors,garage,lot_size,city_index,price
0,1360,2,4,18,1,0,5566,1,248990
1,4272,3,4,29,2,2,2909,0,941769
2,3592,1,4,13,2,1,7364,8,378015
3,966,1,4,19,1,0,3581,0,196887
4,4926,2,4,48,1,1,1691,3,1063441


In [54]:
type(data)

pandas.core.frame.DataFrame

In [55]:
import numpy as np

In [56]:
X = data.drop(columns=['price'])
Y = data[['price']]

In [57]:
type(X)

pandas.core.frame.DataFrame

In [58]:
type(Y)

pandas.core.frame.DataFrame

In [59]:
X_new = X.to_numpy(dtype=np.float32)
Y_new = Y.to_numpy(dtype=np.float32)

In [60]:
X_new

array([[1.360e+03, 2.000e+00, 4.000e+00, ..., 0.000e+00, 5.566e+03,
        1.000e+00],
       [4.272e+03, 3.000e+00, 4.000e+00, ..., 2.000e+00, 2.909e+03,
        0.000e+00],
       [3.592e+03, 1.000e+00, 4.000e+00, ..., 1.000e+00, 7.364e+03,
        8.000e+00],
       ...,
       [2.606e+03, 4.000e+00, 4.000e+00, ..., 1.000e+00, 1.495e+03,
        0.000e+00],
       [4.723e+03, 5.000e+00, 2.000e+00, ..., 0.000e+00, 3.148e+03,
        9.000e+00],
       [3.268e+03, 4.000e+00, 3.000e+00, ..., 2.000e+00, 1.556e+03,
        2.000e+00]], shape=(1000, 8), dtype=float32)

In [61]:
Y_new

array([[2.489900e+05],
       [9.417690e+05],
       [3.780150e+05],
       [1.968870e+05],
       [1.063441e+06],
       [7.620430e+05],
       [3.853100e+05],
       [6.964050e+05],
       [1.075540e+05],
       [2.646830e+05],
       [2.511190e+05],
       [2.788440e+05],
       [5.383440e+05],
       [2.121430e+05],
       [6.913000e+05],
       [5.511200e+05],
       [5.596650e+05],
       [1.030460e+05],
       [2.685180e+05],
       [5.116900e+05],
       [4.384960e+05],
       [4.004780e+05],
       [3.059060e+05],
       [3.080190e+05],
       [1.930710e+05],
       [5.373250e+05],
       [5.487110e+05],
       [2.933890e+05],
       [2.861310e+05],
       [3.066510e+05],
       [7.085990e+05],
       [7.398180e+05],
       [6.493070e+05],
       [1.738080e+05],
       [4.483820e+05],
       [4.368770e+05],
       [1.989510e+05],
       [3.795250e+05],
       [4.026020e+05],
       [1.789350e+05],
       [3.183460e+05],
       [3.936520e+05],
       [3.508920e+05],
       [2.9

In [62]:
from sklearn.model_selection import train_test_split

In [63]:
 x_train, x_test, y_train, y_test = train_test_split(X_new, Y_new, test_size=0.2, random_state=42)

In [64]:
from sklearn.preprocessing import StandardScaler

In [65]:
x_scalar = StandardScaler().fit(x_train)
y_scalar = StandardScaler().fit(y_train)

In [66]:
x_train_scalar = x_scalar.transform(x_train).astype(np.float32)
x_test_scalar = x_scalar.transform(x_test).astype(np.float32)
y_train_scalar = y_scalar.transform(y_train).astype(np.float32)
y_test_scalar = y_scalar.transform(y_test).astype(np.float32)

In [67]:
import torch

In [68]:
x_train_tensor = torch.from_numpy(x_train_scalar)
x_test_tensor = torch.from_numpy(x_test_scalar)
y_train_tensor = torch.from_numpy(y_train_scalar)
y_test_tensor = torch.from_numpy(y_test_scalar)

In [69]:
x_train_tensor.shape

torch.Size([800, 8])

In [70]:
x_train_tensor

tensor([[-0.6277, -0.6820, -0.4487,  ..., -0.0493, -0.0639, -1.5592],
        [ 0.5632,  1.4300, -1.3241,  ...,  1.1831,  0.6207,  0.4829],
        [ 1.3523, -0.6820, -0.4487,  ...,  1.1831, -1.4758,  1.1635],
        ...,
        [-1.0622,  1.4300,  0.4268,  ..., -0.0493, -1.2117, -1.2188],
        [-0.7579,  0.7260,  0.4268,  ..., -1.2817,  0.7054, -1.2188],
        [-1.1589,  0.7260, -1.3241,  ..., -1.2817, -1.1701,  0.8232]])

In [71]:
import torch.nn as nn
import torch.optim as optim

In [72]:
class HousePricePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(8,1),
        )
    def forward(self, x_train_data):
        return self.network(x_train_data)
    
        
        

In [73]:
model = HousePricePredictor()

In [74]:
model

HousePricePredictor(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [75]:
rows = []
for name, param in model.named_parameters():
    rows.append({
        "Name": name,
        "Shape": tuple(param.shape),
        "Trainable": param.requires_grad,
        "Values (first 5)": [round(v.item(), 4) for v in param.data.flatten()[:5]],
        "Grad (first 5)": None if param.grad is None else [round(g.item(), 4) for g in param.grad.flatten()[:5]]
    })

# Convert to DataFrame for clean display
df = pd.DataFrame(rows)
print(df.to_string(index=False))

            Name  Shape  Trainable                          Values (first 5) Grad (first 5)
network.0.weight (1, 8)       True [-0.2935, 0.0363, 0.2779, 0.0855, -0.349]           None
  network.0.bias   (1,)       True                                  [0.1626]           None


In [76]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [77]:
model.train()
epochs = 1000
for epoch in range(1, epochs+1):
    #1. Forward Pass
    y_predictions = model(x_train_tensor)
    #2. Calculate loss
    loss = criterion(y_predictions, y_train_tensor )
    #3. Optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{epochs} | loss: {loss.item():.6f}")

Epoch  100/1000 | loss: 1.497045
Epoch  200/1000 | loss: 1.236239
Epoch  300/1000 | loss: 1.038863
Epoch  400/1000 | loss: 0.883246
Epoch  500/1000 | loss: 0.756616
Epoch  600/1000 | loss: 0.651676
Epoch  700/1000 | loss: 0.564308
Epoch  800/1000 | loss: 0.491911
Epoch  900/1000 | loss: 0.432504
Epoch 1000/1000 | loss: 0.384361


In [78]:
rows = []
for name, param in model.named_parameters():
    rows.append({
        "Name": name,
        "Shape": tuple(param.shape),
        "Trainable": param.requires_grad,
        "Values (first 5)": [round(v.item(), 4) for v in param.data.flatten()[:5]],
        "Grad (first 5)": None if param.grad is None else [round(g.item(), 4) for g in param.grad.flatten()[:5]]
    })

# Convert to DataFrame for clean display
df = pd.DataFrame(rows)
print(df.to_string(index=False))

            Name  Shape  Trainable                           Values (first 5)                            Grad (first 5)
network.0.weight (1, 8)       True [0.4875, 0.0296, 0.0283, -0.0002, -0.0026] [-0.7785, -0.0001, -0.0065, -0.0, 0.0011]
  network.0.bias   (1,)       True                                      [0.0]                                     [0.0]


In [82]:
class HousePricePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(8,16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8,4),
            nn.ReLU(),
            nn.Linear(4,1)      
        )
    def forward(self, x_train_data):
        return self.network(x_train_data)

In [83]:
model = HousePricePredictor()
model

HousePricePredictor(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=4, bias=True)
    (5): ReLU()
    (6): Linear(in_features=4, out_features=1, bias=True)
  )
)

In [84]:
rows = []
for name, param in model.named_parameters():
    rows.append({
        "Name": name,
        "Shape": tuple(param.shape),
        "Trainable": param.requires_grad,
        "Values (first 5)": [round(v.item(), 4) for v in param.data.flatten()[:5]],
        "Grad (first 5)": None if param.grad is None else [round(g.item(), 4) for g in param.grad.flatten()[:5]]
    })

# Convert to DataFrame for clean display
df = pd.DataFrame(rows)
print(df.to_string(index=False))

            Name   Shape  Trainable                            Values (first 5) Grad (first 5)
network.0.weight (16, 8)       True [-0.1054, -0.0225, -0.3054, 0.1215, 0.1471]           None
  network.0.bias   (16,)       True  [-0.1293, 0.0931, -0.088, -0.0596, 0.0042]           None
network.2.weight (8, 16)       True   [0.0189, 0.0645, -0.0792, 0.109, -0.2288]           None
  network.2.bias    (8,)       True  [0.0875, -0.1139, -0.1953, -0.193, 0.2273]           None
network.4.weight  (4, 8)       True [-0.2341, 0.0135, -0.1191, -0.1118, 0.3068]           None
  network.4.bias    (4,)       True         [-0.3491, 0.1132, -0.3368, -0.1488]           None
network.6.weight  (1, 4)       True         [0.2552, -0.3416, -0.3655, -0.3685]           None
  network.6.bias    (1,)       True                                    [0.4335]           None


In [85]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [86]:
model.train()
epochs = 1000
for epoch in range(1, epochs+1):
    #1. Forward Pass
    y_predictions = model(x_train_tensor)
    #2. Calculate loss
    loss = criterion(y_predictions, y_train_tensor )
    #3. Optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{epochs} | loss: {loss.item():.6f}")

Epoch  100/1000 | loss: 0.650060
Epoch  200/1000 | loss: 0.360427
Epoch  300/1000 | loss: 0.322940
Epoch  400/1000 | loss: 0.297484
Epoch  500/1000 | loss: 0.278508
Epoch  600/1000 | loss: 0.262340
Epoch  700/1000 | loss: 0.248153
Epoch  800/1000 | loss: 0.234795
Epoch  900/1000 | loss: 0.224449
Epoch 1000/1000 | loss: 0.216677


In [87]:
rows = []
for name, param in model.named_parameters():
    rows.append({
        "Name": name,
        "Shape": tuple(param.shape),
        "Trainable": param.requires_grad,
        "Values (first 5)": [round(v.item(), 4) for v in param.data.flatten()[:5]],
        "Grad (first 5)": None if param.grad is None else [round(g.item(), 4) for g in param.grad.flatten()[:5]]
    })

# Convert to DataFrame for clean display
df = pd.DataFrame(rows)
print(df.to_string(index=False))

            Name   Shape  Trainable                            Values (first 5)                               Grad (first 5)
network.0.weight (16, 8)       True  [-0.2501, 0.0707, -0.4031, 0.1758, 0.0782]   [-0.0028, 0.0012, 0.0006, 0.0013, -0.0004]
  network.0.bias   (16,)       True     [0.1685, 0.2289, 0.329, 0.0777, 0.0525]  [-0.0002, -0.0035, -0.0017, 0.0008, 0.0019]
network.2.weight (8, 16)       True   [0.0976, -0.027, 0.1871, 0.0155, -0.3799] [-0.0001, -0.0003, -0.0014, 0.0006, -0.0002]
  network.2.bias    (8,)       True [0.1112, -0.1139, -0.0245, -0.0156, 0.2816]      [-0.0005, 0.0, 0.0005, -0.001, -0.0025]
network.4.weight  (4, 8)       True  [-0.2341, 0.0135, -0.1191, -0.158, 0.2663]                    [0.0, 0.0, 0.0, 0.0, 0.0]
  network.4.bias    (4,)       True          [-0.387, 0.1632, -0.3368, -0.0944]                  [0.0, -0.0033, 0.0, -0.002]
network.6.weight  (1, 4)       True         [0.2244, -0.5572, -0.3655, -0.6125]                   [0.0, 0.0049, 0.0, 0.0052]


In [90]:
model.eval()
with torch.no_grad():
    y_pred_test = model(x_test_tensor)
    y_pred_numpy = y_pred_test.numpy()


In [92]:
y_pred_numpy

array([[ 0.98152447],
       [-0.3841548 ],
       [ 0.98152447],
       [ 0.98152447],
       [ 0.43945575],
       [ 0.3888625 ],
       [ 0.98152447],
       [ 0.8514563 ],
       [ 0.58532447],
       [ 0.98152447],
       [ 0.33755475],
       [ 0.04713219],
       [-1.4194624 ],
       [ 0.98152447],
       [-0.40766978],
       [-1.1224258 ],
       [-0.49737358],
       [-0.7015606 ],
       [ 0.67900217],
       [-1.2616932 ],
       [ 0.5045682 ],
       [-0.39698195],
       [ 0.43568552],
       [ 0.6203736 ],
       [-0.35888457],
       [-0.96236455],
       [-0.6387472 ],
       [-0.65050364],
       [-1.3197334 ],
       [-0.69568586],
       [-1.2091508 ],
       [ 0.98152447],
       [ 0.7174822 ],
       [-0.9232702 ],
       [-0.6736449 ],
       [ 0.88796985],
       [-0.6591505 ],
       [-0.2868476 ],
       [-1.1116142 ],
       [-1.4015899 ],
       [-1.1953816 ],
       [-1.1954324 ],
       [-0.46515822],
       [-1.0857034 ],
       [ 0.98152447],
       [ 0

In [95]:
y_pred = y_scalar.inverse_transform(y_pred_numpy).flatten()

In [96]:
y_pred

array([660632.44 , 371047.03 , 660632.44 , 660632.44 , 545689.5  ,
       534961.44 , 660632.44 , 633052.1  , 576620.25 , 660632.44 ,
       524081.9  , 462499.28 , 151515.25 , 660632.44 , 366060.8  ,
       214500.38 , 347039.56 , 303742.72 , 596484.1  , 184969.4  ,
       559496.25 , 368327.1  , 544890.06 , 584052.25 , 376405.44 ,
       248440.56 , 317062.   , 314569.12 , 172662.31 , 304988.44 ,
       196110.78 , 660632.44 , 604643.6  , 256730.31 , 309662.12 ,
       640794.7  , 312735.56 , 391680.53 , 216792.92 , 155305.03 ,
       199030.47 , 199019.69 , 353870.66 , 222287.19 , 660632.44 ,
       660632.44 , 660632.44 , 648838.44 , 660632.44 , 468939.4  ,
       200759.31 , 522425.12 , 645637.3  , 541549.9  , 371312.84 ,
       660632.44 , 105454.375, 208562.34 , 590610.06 , 128184.56 ,
       660632.44 , 220238.69 , 205529.88 , 355395.06 , 457063.22 ,
       397450.5  , 289595.5  , 552269.3  , 660632.44 , 542969.5  ,
       328363.62 , 659605.8  , 254645.61 , 417570.16 , 570475.

In [97]:
y_test

array([[746264.],
       [349011.],
       [853367.],
       [784465.],
       [342987.],
       [563293.],
       [688837.],
       [849834.],
       [432060.],
       [683096.],
       [678074.],
       [366480.],
       [155385.],
       [861873.],
       [499054.],
       [145459.],
       [273237.],
       [356791.],
       [673463.],
       [236566.],
       [431443.],
       [242937.],
       [550780.],
       [719810.],
       [281586.],
       [171912.],
       [494121.],
       [327919.],
       [227518.],
       [365159.],
       [311099.],
       [683841.],
       [619337.],
       [247016.],
       [331006.],
       [477185.],
       [403971.],
       [424737.],
       [165438.],
       [303278.],
       [313692.],
       [190874.],
       [387074.],
       [138589.],
       [868454.],
       [941226.],
       [609345.],
       [683120.],
       [840670.],
       [471042.],
       [179110.],
       [659637.],
       [711070.],
       [630874.],
       [308019.],
       [70